# 📊 Paper 170 Analysis: A Machine Learning View on Momentum and Reversal Trading

## 🎯 Research Goal
Reproduce the research from "A Machine Learning View on Momentum and Reversal Trading" paper using real S&P 500 data instead of Chinese market data.

## 📋 Two-Phase Approach
- **Phase 1**: Data Collection & Persistence (Cells 1-8)
- **Phase 2**: Feature Engineering & ML (Cells 9-16)

---

# 🚀 Phase 1: Data Collection & Persistence


## Step 1: Dependencies & Setup

Import all required libraries and set up configurations for data collection.


In [7]:
# Core data manipulation and analysis
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Financial data fetching
import yfinance as yf
import time
from datetime import datetime, timedelta

# Technical analysis
from scipy import stats
from scipy.stats import kurtosis, skew

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Data persistence
import pickle
import json
import os

# Progress tracking
from tqdm import tqdm

# Set up data directory
data_dir = 'data/research'
os.makedirs(data_dir, exist_ok=True)

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Docker network configuration (if running in Docker)
import os
if os.path.exists('/.dockerenv'):
    os.environ['PYTHONHTTPSVERIFY'] = '0'
    os.environ['CURL_CA_BUNDLE'] = ''
    print("🐳 Docker environment detected - network settings applied")

print("✅ Dependencies loaded successfully!")
print(f"📁 Data directory: {data_dir}")
print(f"📅 Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

🐳 Docker environment detected - network settings applied
✅ Dependencies loaded successfully!
📁 Data directory: data/research
📅 Current time: 2025-10-22 19:27:11


## Step 2: S&P 500 Ticker Collection

Get a curated list of 250+ major S&P 500 tickers for comprehensive market coverage.


In [8]:
def get_sp500_tickers():
    """Get curated list of 250+ major S&P 500 tickers"""
    print("📥 Collecting S&P 500 ticker symbols...")
    
    # Curated list of major S&P 500 tickers (250+ companies)
    sp500_tickers = [
        # Technology (50+)
        'AAPL', 'MSFT', 'GOOGL', 'GOOG', 'AMZN', 'TSLA', 'META', 'NVDA', 'NFLX', 'ADBE',
        'CRM', 'ORCL', 'INTC', 'AMD', 'CSCO', 'IBM', 'QCOM', 'TXN', 'AVGO', 'AMAT',
        'MU', 'ADI', 'LRCX', 'KLAC', 'MCHP', 'SNPS', 'CDNS', 'ANSS', 'FTNT', 'PANW',
        'CRWD', 'ZS', 'OKTA', 'DDOG', 'NET', 'SNOW', 'PLTR', 'RBLX', 'U', 'TWLO',
        'ZM', 'DOCU', 'SPLK', 'WDAY', 'NOW', 'TEAM', 'VEEV', 'MDB', 'ESTC', 'DDOG',
        'CFLT', 'FROG', 'PATH', 'BILL', 'AI', 'SMCI', 'ARM', 'ARMK', 'ARW', 'ASML',
        
        # Healthcare (40+)
        'JNJ', 'PFE', 'UNH', 'ABBV', 'MRK', 'TMO', 'ABT', 'DHR', 'BMY', 'AMGN',
        'GILD', 'BIIB', 'REGN', 'VRTX', 'ILMN', 'MRNA', 'BNTX', 'ZTS', 'SYK', 'ISRG',
        'MDT', 'BSX', 'EW', 'DXCM', 'TECH', 'IQV', 'A', 'WAT', 'PKI', 'WST',
        'TMO', 'DHR', 'A', 'WAT', 'PKI', 'WST', 'TMO', 'DHR', 'A', 'WAT',
        
        # Financial Services (35+)
        'JPM', 'BAC', 'WFC', 'GS', 'MS', 'C', 'AXP', 'BLK', 'SPGI', 'MCO',
        'V', 'MA', 'PYPL', 'COF', 'USB', 'PNC', 'TFC', 'BK', 'STT', 'NTRS',
        'SCHW', 'ICE', 'CME', 'NDAQ', 'MKTX', 'FIS', 'FISV', 'GPN', 'JKHY', 'FLT',
        'WU', 'TRV', 'ALL', 'PGR', 'AON', 'MMC', 'AFL', 'PRU', 'MET', 'AIG',
        
        # Consumer Discretionary (30+)
        'HD', 'MCD', 'NKE', 'SBUX', 'LOW', 'TJX', 'ROST', 'TGT', 'WMT', 'COST',
        'AMZN', 'TSLA', 'F', 'GM', 'FORD', 'TM', 'HMC', 'NIO', 'XPEV', 'LI',
        'BABA', 'JD', 'PDD', 'BIDU', 'NTES', 'TME', 'VIPS', 'YMM', 'DIDI', 'GRAB',
        
        # Consumer Staples (25+)
        'PG', 'KO', 'PEP', 'WMT', 'COST', 'CL', 'KMB', 'CHD', 'CLX', 'GIS',
        'K', 'CPB', 'HSY', 'MKC', 'SJM', 'CAG', 'HRL', 'TSN', 'KHC', 'MDLZ',
        'PM', 'MO', 'BTI', 'IMB', 'UL', 'NVS', 'ASML', 'SAP', 'TM', 'HMC',
        
        # Industrial (30+)
        'BA', 'CAT', 'GE', 'HON', 'MMM', 'UPS', 'FDX', 'LMT', 'RTX', 'NOC',
        'GD', 'TDG', 'LHX', 'TDY', 'TDG', 'LHX', 'TDY', 'TDG', 'LHX', 'TDY',
        'EMR', 'ETN', 'ITW', 'PH', 'ROK', 'SWK', 'TXT', 'DOV', 'FTV', 'IEX',
        
        # Energy (20+)
        'XOM', 'CVX', 'COP', 'EOG', 'SLB', 'OXY', 'PXD', 'MPC', 'VLO', 'PSX',
        'KMI', 'WMB', 'EPD', 'OKE', 'ET', 'ENB', 'TRP', 'PPL', 'DUK', 'SO',
        
        # Materials (15+)
        'LIN', 'APD', 'SHW', 'ECL', 'DD', 'DOW', 'FCX', 'NEM', 'GOLD', 'AA',
        'X', 'CLF', 'NUE', 'STLD', 'CMC', 'RS', 'VMC', 'MLM', 'EXP', 'SUM',
        
        # Utilities (15+)
        'NEE', 'DUK', 'SO', 'D', 'EXC', 'AEP', 'XEL', 'PEG', 'ES', 'EIX',
        'SRE', 'WEC', 'AWK', 'LNT', 'CNP', 'ED', 'PCG', 'ETR', 'FE', 'AEE',
        
        # Real Estate (10+)
        'AMT', 'PLD', 'CCI', 'EQIX', 'PSA', 'EXR', 'AVB', 'EQR', 'MAA', 'UDR',
        'ESS', 'CPT', 'AIV', 'BXP', 'VTR', 'WELL', 'PEAK', 'HCP', 'VTR', 'WELL',
        
        # Communication Services (15+)
        'GOOGL', 'GOOG', 'META', 'NFLX', 'DIS', 'CMCSA', 'VZ', 'T', 'CHTR', 'TMUS',
        'TWTR', 'SNAP', 'PINS', 'SPOT', 'MTCH', 'LYFT', 'UBER', 'DASH', 'ABNB', 'BKNG'
    ]
    
    # Remove duplicates and sort
    sp500_tickers = sorted(list(set(sp500_tickers)))
    
    print(f"✅ Collected {len(sp500_tickers)} S&P 500 tickers")
    print(f"📊 Sample tickers: {sp500_tickers[:10]}...")
    
    return sp500_tickers

# Get S&P 500 tickers
sp500_tickers = get_sp500_tickers()

# Save tickers for reference
tickers_file = f'{data_dir}/sp500_tickers.json'
with open(tickers_file, 'w') as f:
    json.dump(sp500_tickers, f, indent=2)

print(f"💾 Saved tickers to: {tickers_file}")


📥 Collecting S&P 500 ticker symbols...
✅ Collected 298 S&P 500 tickers
📊 Sample tickers: ['A', 'AA', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ADBE', 'ADI', 'AEE', 'AEP']...
💾 Saved tickers to: data/research/sp500_tickers.json


## Step 4: Real Stock Data Collection

Fetch real historical stock data (OHLCV) for all S&P 500 tickers with proper error handling and rate limiting.


In [9]:
def fetch_real_stock_data(tickers, start_date='2015-01-01', end_date='2024-12-31'):
    """Fetch real stock data for S&P 500 tickers"""
    print(f"📥 Fetching real stock data for {len(tickers)} S&P 500 stocks...")
    print(f"📅 Date range: {start_date} to {end_date}")
    
    all_data = []
    successful_tickers = []
    failed_tickers = []
    
    for i, ticker in enumerate(tqdm(tickers, desc="Fetching data")):
        print(f"📊 Fetching {ticker} ({i+1}/{len(tickers)})...")
        
        try:
            # Create ticker object
            stock = yf.Ticker(ticker)
            
            # Fetch historical data
            hist = stock.history(start=start_date, end=end_date)
            
            if hist.empty:
                print(f"    ⚠️  {ticker}: No data available")
                failed_tickers.append(ticker)
                continue
            
            # Reset index to get date as column
            hist = hist.reset_index()
            
            # Add symbol column
            hist['symbol'] = ticker
            
            # Rename columns to lowercase
            hist.columns = [col.lower() for col in hist.columns]
            
            # Ensure we have required columns
            required_cols = ['date', 'open', 'high', 'low', 'close', 'volume', 'symbol']
            if all(col in hist.columns for col in required_cols):
                all_data.append(hist[required_cols])
                successful_tickers.append(ticker)
                print(f"    ✅ {ticker}: {len(hist)} records")
            else:
                print(f"    ⚠️  {ticker}: Missing required columns")
                failed_tickers.append(ticker)
            
            # Rate limiting
            time.sleep(0.1)
            
        except Exception as e:
            print(f"    ❌ {ticker}: {str(e)}")
            failed_tickers.append(ticker)
            continue
    
    if not all_data:
        print("❌ No data retrieved from any ticker")
        return None
    
    # Combine all data
    df = pd.concat(all_data, ignore_index=True)
    
    # Sort by symbol and date
    df = df.sort_values(['symbol', 'date']).reset_index(drop=True)
    
    print(f"\n✅ Data collection complete!")
    print(f"   📊 Total records: {len(df):,}")
    print(f"   📈 Successful tickers: {len(successful_tickers)}")
    print(f"   ❌ Failed tickers: {len(failed_tickers)}")
    print(f"   📅 Date range: {df['date'].min()} to {df['date'].max()}")
    print(f"   💰 Symbols: {df['symbol'].nunique()}")
    
    if failed_tickers:
        print(f"   ⚠️  Failed tickers: {failed_tickers[:10]}{'...' if len(failed_tickers) > 10 else ''}")
    
    return df, successful_tickers, failed_tickers

# Fetch stock data
stock_data, successful_tickers, failed_tickers = fetch_real_stock_data(sp500_tickers)

if stock_data is not None:
    print(f"\n📊 Sample of collected data:")
    print(stock_data.head())
    
    print(f"\n📋 Data summary:")
    print(f"   Shape: {stock_data.shape}")
    print(f"   Columns: {list(stock_data.columns)}")
    print(f"   Date range: {stock_data['date'].min()} to {stock_data['date'].max()}")
    print(f"   Symbols: {stock_data['symbol'].nunique()}")
else:
    print("❌ Failed to collect stock data")


📥 Fetching real stock data for 298 S&P 500 stocks...
📅 Date range: 2015-01-01 to 2024-12-31


Fetching data:   0%|          | 0/298 [00:00<?, ?it/s]

📊 Fetching A (1/298)...


Fetching data:   0%|          | 1/298 [00:00<03:11,  1.55it/s]

    ✅ A: 2515 records
📊 Fetching AA (2/298)...


...

    ✅ ZM: 1435 records
📊 Fetching ZS (297/298)...


Fetching data: 100%|█████████▉| 297/298 [01:57<00:00,  2.92it/s]

    ✅ ZS: 1709 records
📊 Fetching ZTS (298/298)...


Fetching data: 100%|██████████| 298/298 [01:58<00:00,  2.52it/s]

    ✅ ZTS: 2515 records

✅ Data collection complete!
   📊 Total records: 669,825
   📈 Successful tickers: 285
   ❌ Failed tickers: 13
   📅 Date range: 2015-01-02 00:00:00-05:00 to 2024-12-30 00:00:00-05:00
   💰 Symbols: 285
   ⚠️  Failed tickers: ['ANSS', 'DIDI', 'FISV', 'FLT', 'GOLD', 'HCP', 'PEAK', 'PKI', 'PXD', 'SPLK']...

📊 Sample of collected data:
                       date       open       high        low      close  \
0 2015-01-02 00:00:00-05:00  37.688386  37.807365  36.947064  37.120956   
1 2015-01-05 00:00:00-05:00  36.901310  37.029439  36.333880  36.425400   
2 2015-01-06 00:00:00-05:00  36.434540  36.626733  35.711522  35.857956   
3 2015-01-07 00:00:00-05:00  36.169143  36.434555  35.958645  36.333881   
4 2015-01-08 00:00:00-05:00  36.828074  37.505327  36.773160  37.422958   

    volume symbol  
0  1529200      A  
1  2041800      A  
2  2080600      A  
3  3359700      A  
4  2116300      A  

📋 Data summary:
   Shape: (669825, 7)
   Columns: ['date', 'open', 'high

## Step 5: SEC Quarterly Fundamental Data

Fetch quarterly fundamental data from SEC using point-in-time accuracy (filed dates) to avoid lookahead bias.


In [10]:
# Helper function to get CIK mappings from SEC
def get_sec_ticker_cik_map():
    """
    Fetch the complete ticker->CIK mapping from SEC
    Returns: dict of {ticker: cik}
    """
    import requests
    
    SEC_BASE = "https://www.sec.gov"
    HEADERS = {
        "User-Agent": "Noam Polak noampolak@gmail.com",  # Required by SEC
        "Accept-Encoding": "gzip, deflate",
        "Host": "www.sec.gov"
    }
    
    url = "https://www.sec.gov/files/company_tickers.json"
    
    print("📥 Fetching SEC ticker-CIK mapping...")
    response = requests.get(url, headers=HEADERS)
    
    if response.status_code != 200:
        raise Exception(f"Failed to fetch CIK map: {response.status_code}")
    
    data = response.json()
    
    # Convert to {ticker: CIK} dict
    ticker_cik_map = {}
    for entry in data.values():
        ticker = entry['ticker']
        cik = str(entry['cik_str']).zfill(10)  # Pad to 10 digits
        ticker_cik_map[ticker] = cik
    
    print(f"✅ Found {len(ticker_cik_map)} ticker-CIK mappings")
    return ticker_cik_map

print("✅ CIK mapping function loaded")


✅ CIK mapping function loaded


In [11]:
# ===== SEC quarterly fundamentals (2014-12-31 -> today), robust FY/CY frames fallback, cached =====
# You provide:
#   - tickers: list[str], e.g. ["AAPL","MSFT"]
#   - cik_map: dict[ticker]->10-digit CIK, e.g. {"AAPL":"0000320193", ...}
#   - price_df: DataFrame with ['ticker','period_end','price_close', ...] where period_end is quarter-end date
#
# Output columns (per quarter):
#   Base: revenue, net_income, eps_diluted, shares_out, equity, operating_cash_flow, dividends_per_share
#   Prices: price_close
#   Point-in-time: market_cap, pe_q, pb, book_to_market
#   TTM sums: revenue_ttm, operating_cash_flow_ttm, eps_diluted_ttm, dividends_per_share_ttm
#   TTM ratios: pe_ttm, ps_ttm, pcf_ttm, dividend_yield_ttm

import os, re, time, json, hashlib, warnings, datetime as dt
import pandas as pd
import numpy as np
import requests

warnings.filterwarnings("ignore")

# ----------------- Config -----------------
USER_AGENT = "Noam Polak (noampolak@gmail.com)"      # <— real name + email required by SEC
START_DATE = pd.Timestamp("2014-09-30")              # inclusive
DATE_BUFFER_DAYS = 14                                # accept fiscal quarter ends slightly before 12/31
USE_FRAMES_FALLBACK = True                           # for AAPL & others this is needed
SLEEP_SEC = 0.08                                     # pacing when hitting frames
CACHE_DIR = "./sec_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

SEC_BASE = "https://data.sec.gov"
HEADERS = {"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate", "Host": "data.sec.gov"}
QFPS = {"Q1","Q2","Q3","Q4"}

# ----------------- GAAP tag sets -----------------
TAG_REVENUE = ["RevenueFromContractWithCustomerExcludingAssessedTax", "SalesRevenueNet", "Revenues"]
TAG_NET_INCOME = ["NetIncomeLoss"]
TAG_EPS_DILUTED = ["EarningsPerShareDiluted", "EarningsPerShareBasicAndDiluted"]
TAG_SHARES = ["CommonStockSharesOutstanding",
              "WeightedAverageNumberOfDilutedSharesOutstanding",
              "WeightedAverageNumberOfSharesOutstandingDiluted"]
TAG_EQUITY = ["StockholdersEquity",
              "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
              "CommonStockholdersEquity"]
TAG_CFO = ["NetCashProvidedByUsedInOperatingActivities",
           "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"]
TAG_DPS = ["CommonStockDividendsPerShareDeclared", "CommonStockDividendsPerShareCashPaid"]

# ----------------- Cache & HTTP helpers -----------------
_session = requests.Session()
_session.headers.update(HEADERS)

def _cache_path(url: str) -> str:
    h = hashlib.md5(url.encode("utf-8")).hexdigest()
    return os.path.join(CACHE_DIR, f"{h}.json")

def fetch_json(url, params=None, use_cache=True, sleep=SLEEP_SEC):
    path = _cache_path(url + ("?" + json.dumps(params, sort_keys=True) if params else ""))
    if use_cache and os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    for i in range(3):
        r = _session.get(url, params=params, timeout=30)
        if r.status_code == 200:
            data = r.json()
            if use_cache:
                with open(path, "w") as f:
                    json.dump(data, f)
            return data
        # frames often 404 for a given combination; treat as empty instead of crashing
        if r.status_code == 404 and "/api/xbrl/frames/" in url:
            return {}
        time.sleep(0.4 * (i + 1))
    r.raise_for_status()
    


def get_company_facts(cik: str):
    url = f"{SEC_BASE}/api/xbrl/companyfacts/CIK{cik}.json"
    return fetch_json(url, use_cache=True, sleep=0)  # cached; 1 call per ticker

def _gaap_root(facts_dict):
    """Return the dict that holds us-gaap facts inside the CompanyFacts payload."""
    return (facts_dict or {}).get("facts", {}).get("us-gaap", {})

# ----------------- CompanyFacts quarterly extractor (robust) -----------------
def _extract_quarterly_series_robust(facts: dict, tag_names, unit_priority=None):
    """
    Extract quarterly rows for the first tag that yields data in companyfacts.
    Quarterly if fp ∈ {Q1..Q4} OR frame contains 'Q1..Q4'. Try ALL units if unit_priority None.
    Returns df: end, filed, fy, fp, frame, val
    """
    gaap = _gaap_root(facts)
    for tag in tag_names:
        node = gaap.get(tag)
        if not node:
            continue
        units_dict = node.get("units", {})
        unit_list = unit_priority or tuple(units_dict.keys())
        for unit in unit_list:
            vals = units_dict.get(unit, [])
            if not vals:
                continue
            rows = []
            for v in vals:
                fp = v.get("fp")
                frame = v.get("frame", "")
                looks_quarterly = (fp in QFPS) or (re.search(r"Q[1-4]", str(frame)) is not None)
                if not looks_quarterly:
                    continue
                end = v.get("end")
                if not end:
                    continue
                end_dt = pd.to_datetime(end, errors="coerce")
                if pd.isna(end_dt) or end_dt < (START_DATE - pd.Timedelta(days=DATE_BUFFER_DAYS)):
                    continue
                rows.append({
                    "end": end_dt.normalize(),
                    "filed": pd.to_datetime(v.get("filed"), errors="coerce"),
                    "fy": v.get("fy"),
                    "fp": fp,
                    "frame": frame,
                    "val": v.get("val")
                })
            if rows:
                df = pd.DataFrame(rows).dropna(subset=["end"])
                df = (df.sort_values(["end","filed"])
                        .groupby("end", as_index=False)
                        .tail(1)
                        .sort_values("end"))
                return df
    return pd.DataFrame(columns=["end","filed","fy","fp","frame","val"])


# ----------------- Frames fallback (FY + CY; instant vs duration) -----------------
def _missing_frames(start_date: pd.Timestamp, have_ends: pd.Series):
    want = []
    start = (start_date - pd.Timedelta(days=DATE_BUFFER_DAYS)).to_period("Q")
    end   = pd.Timestamp.today().to_period("Q")
    have_periods = have_ends.dt.to_period("Q") if not have_ends.empty else pd.Series([], dtype="period[Q-DEC]")
    p = start
    while p <= end:
        if have_periods.empty or not (have_periods == p).any():
            want.append((p.year, p.quarter))
        p += 1
    return want

def _frames_quarter_series(cik: str, tag: str, unit: str, quarters: list, kind: str):
    """
    quarters: list of (year, qnum)
    kind: 'instant' (balance sheet) or 'duration' (income/cashflow/per-share)
    Tries FY… first (for fiscal reporters like AAPL), then CY… .
    """
    rows = []
    for year, q in quarters:
        if kind == "instant":
            variants = [f"FY{year}Q{q}I", f"CY{year}Q{q}I"]
        else:
            variants = [f"FY{year}Q{q}", f"FY{year}Q{q}YTD", f"CY{year}Q{q}", f"CY{year}Q{q}YTD"]
        got = False
        for frame_id in variants:
            url = f"{SEC_BASE}/api/xbrl/frames/us-gaap/{tag}/{unit}/{frame_id}"
            data = fetch_json(url, use_cache=True)
            if not data or "data" not in data:
                continue
            for row in data["data"]:
                if str(row.get("cik")).zfill(10) != str(cik):
                    continue
                end_dt = pd.to_datetime(row.get("end"), errors="coerce")
                if pd.isna(end_dt) or end_dt < (START_DATE - pd.Timedelta(days=DATE_BUFFER_DAYS)):
                    continue
                rows.append({
                    "end": end_dt.normalize(),
                    "filed": pd.to_datetime(row.get("filed"), errors="coerce"),
                    "fy": row.get("fy"),
                    "fp": row.get("fp"),
                    "frame": row.get("frame"),
                    "val": row.get("val")
                })
                got = True
            if got:
                break
    if not rows:
        return pd.DataFrame(columns=["end","filed","fy","fp","frame","val"])
    df = (pd.DataFrame(rows)
            .sort_values(["end","filed"])
            .groupby("end", as_index=False).tail(1)
            .sort_values("end"))
    return df

# ----------------- Build quarterly facts (companyfacts + frames) -----------------
def _pull_quarterly_facts(facts: dict, cik: str):
    def get_series(tags, prefer_units=None, unit_for_frames="USD", kind="duration"):
        s = _extract_quarterly_series_robust(facts, tags, unit_priority=prefer_units)
        if not USE_FRAMES_FALLBACK:
            return s
        have = s["end"] if not s.empty else pd.Series([], dtype="datetime64[ns]")
        missing = _missing_frames(START_DATE, have)
        if not missing:
            return s
        first_tag = tags[0]
        sf = _frames_quarter_series(cik, first_tag, unit_for_frames, missing, kind)
        return (pd.concat([s, sf], ignore_index=True)
                  .drop_duplicates(subset=["end"])
                  .sort_values("end"))

    # duration (income/cash flow / per-share)
    rev = get_series(TAG_REVENUE, kind="duration")
    ni  = get_series(TAG_NET_INCOME, kind="duration")
    eps = get_series(TAG_EPS_DILUTED, prefer_units=("USD/shares","USD","pure"), unit_for_frames="USD", kind="duration")
    cfo = get_series(TAG_CFO, kind="duration")
    dps = get_series(TAG_DPS, prefer_units=("USD/shares","USD"), unit_for_frames="USD", kind="duration")
    # instant (balance sheet)
    sh  = get_series(TAG_SHARES, prefer_units=("shares","pure"), unit_for_frames="shares", kind="instant")
    eq  = get_series(TAG_EQUITY, kind="instant")

    out = pd.DataFrame({"end": pd.to_datetime([])})
    for df_, name in [(rev,"revenue"), (ni,"net_income"), (eps,"eps_diluted"),
                      (sh,"shares_out"), (eq,"equity"),
                      (cfo,"operating_cash_flow"), (dps,"dividends_per_share")]:
        if not df_.empty:
            out = out.merge(df_[["end","val"]].rename(columns={"val": name}),
                            on="end", how="outer")

    if out.empty:
        return out

    base = rev if not rev.empty else (ni if not ni.empty else (eps if not eps.empty else None))
    if base is not None and not base.empty:
        base_cols = [c for c in ["end","fy","fp","frame"] if c in base.columns]
        out = out.merge(base[base_cols], on="end", how="left")
    else:
        out["fy"]=np.nan; out["fp"]=np.nan; out["frame"]=np.nan

    out = out[out["end"] >= (START_DATE - pd.Timedelta(days=DATE_BUFFER_DAYS))].copy()
    out = out.sort_values("end").reset_index(drop=True)
    out["year"] = out["end"].dt.year
    out["quarter"] = out["end"].dt.quarter
    return out

# ----------------- Ratios / merge with your prices -----------------
def safe_divide(numerator, denominator, default=np.nan):
    """Safely divide two arrays/series, returning default for division by zero"""
    with np.errstate(divide='ignore', invalid='ignore'):
        result = numerator / denominator
        # Handle both numpy arrays and pandas Series
        if isinstance(result, pd.Series):
            result = result.replace([np.inf, -np.inf], default)
        else:
            result = np.where(np.isfinite(result), result, default)
    return result

def compute_quarterly_and_ttm(df_q: pd.DataFrame, price_df: pd.DataFrame, ticker: str):
    if df_q.empty:
        return df_q
    df = df_q.copy()
    
    # Configure numpy to convert division by zero to NaN instead of raising errors
    np.seterr(divide='ignore', invalid='ignore')

    px_cols = ["ticker","period_end","price_close"]
    extra_px_cols = [c for c in price_df.columns if c not in px_cols]
    merged = df.merge(
        price_df[["ticker","period_end","price_close"] + extra_px_cols]
            .rename(columns={"period_end":"end"}),
        on="end", how="left"
    )
    merged["ticker"] = ticker

    # Calculate shares_final with safety checks for missing columns
    if "shares_out" in merged.columns and "net_income" in merged.columns and "eps_diluted" in merged.columns:
        # Use safe_divide for computing shares from net_income / eps
        computed_shares = safe_divide(merged["net_income"], merged["eps_diluted"])
        merged["shares_final"] = np.where(
            merged["shares_out"].isna() & merged["net_income"].notna() & merged["eps_diluted"].notna(),
            computed_shares,
            merged["shares_out"]
        )
    elif "shares_out" in merged.columns:
        merged["shares_final"] = merged["shares_out"]
    else:
        merged["shares_final"] = np.nan

    merged["market_cap"] = merged["price_close"] * merged["shares_final"]
    
    if "eps_diluted" in merged.columns:
        merged["pe_q"] = safe_divide(merged["price_close"], merged["eps_diluted"])
    else:
        merged["pe_q"] = np.nan
    
    if "equity" in merged.columns:
        merged["pb"] = safe_divide(merged["market_cap"], merged["equity"])
        merged["book_to_market"] = safe_divide(merged["equity"], merged["market_cap"])
    else:
        merged["pb"] = np.nan
        merged["book_to_market"] = np.nan

    merged = merged.sort_values("end")
    for col in ["revenue","operating_cash_flow","net_income","eps_diluted","dividends_per_share"]:
        if col in merged.columns:
            merged[f"{col}_ttm"] = merged[col].rolling(window=4, min_periods=4).sum()
        else:
            merged[f"{col}_ttm"] = np.nan

    # Calculate TTM ratios with safe division
    if "eps_diluted_ttm" in merged.columns:
        merged["pe_ttm"] = safe_divide(merged["price_close"], merged["eps_diluted_ttm"])
    else:
        merged["pe_ttm"] = np.nan
    
    if "revenue_ttm" in merged.columns:
        merged["ps_ttm"] = safe_divide(merged["market_cap"], merged["revenue_ttm"])
    else:
        merged["ps_ttm"] = np.nan
    
    if "operating_cash_flow_ttm" in merged.columns:
        merged["pcf_ttm"] = safe_divide(merged["market_cap"], merged["operating_cash_flow_ttm"])
    else:
        merged["pcf_ttm"] = np.nan
    
    if "dividends_per_share_ttm" in merged.columns:
        merged["dividend_yield_ttm"] = safe_divide(merged["dividends_per_share_ttm"], merged["price_close"])
    else:
        merged["dividend_yield_ttm"] = np.nan

    # Select only columns that exist in merged dataframe
    desired_cols = [
        "ticker","end","fy","fp","frame","year","quarter","price_close",
        "revenue","net_income","eps_diluted","shares_out","equity","operating_cash_flow","dividends_per_share",
        "market_cap","pe_q","pb","book_to_market","shares_final",
        "revenue_ttm","operating_cash_flow_ttm","eps_diluted_ttm","dividends_per_share_ttm",
        "pe_ttm","ps_ttm","pcf_ttm","dividend_yield_ttm"
    ] + extra_px_cols
    
    # Only include columns that actually exist
    cols = [c for c in desired_cols if c in merged.columns]
    return merged[cols]

# ----------------- Public function -----------------
def fetch_quarterly_fundamentals(ticker: str, cik: str, price_df: pd.DataFrame) -> pd.DataFrame:
    cik = str(int(cik)).zfill(10)
    facts = get_company_facts(cik)           # cached; 1 call per ticker
    df_q = _pull_quarterly_facts(facts, cik) # companyfacts + FY/CY frames fallback
    if df_q.empty:
        return df_q
    df_q = df_q[df_q["end"] >= START_DATE].copy()
    return compute_quarterly_and_ttm(df_q, price_df[price_df["ticker"] == ticker].copy(), ticker)

# tickers = ["AAPL","MSFT"]
# cik_map = {"AAPL":"0000320193","MSFT":"0000789019"}
# out = pd.concat([fetch_quarterly_fundamentals(t, cik_map[t], price_df) for t in tickers], ignore_index=True)
# out.tail(6)
# out.to_csv("sp500_quarterly_fundamentals.csv", index=False)


In [12]:
def add_sec_quarterly_fundamentals(df):
    """Add SEC quarterly fundamental data using filed dates for point-in-time accuracy"""
    print("🚀 Adding SEC quarterly fundamental data...")
    
    if df is None or len(df) == 0:
        print("❌ No data available for fundamental data addition")
        return df
    
    # Get unique symbols
    symbols = df['symbol'].unique()
    print(f"📈 Adding fundamentals for {len(symbols)} stocks...")
    
    # Ensure date column is datetime and timezone-naive
    df['date'] = pd.to_datetime(df['date'])
    if df['date'].dt.tz is not None:
        df['date'] = df['date'].dt.tz_localize(None)
    
    # Step 1: Get CIK mappings for all tickers
    print("\n📥 Fetching CIK mappings from SEC...")
    full_cik_map = get_sec_ticker_cik_map()
    
    # Filter to our tickers
    cik_map = {}
    missing_ciks = []
    for symbol in symbols:
        if symbol in full_cik_map:
            cik_map[symbol] = full_cik_map[symbol]
        else:
            missing_ciks.append(symbol)
    
    print(f"   ✅ Found CIKs for {len(cik_map)}/{len(symbols)} tickers")
    if missing_ciks:
        print(f"   ⚠️  Missing CIK for {len(missing_ciks)} tickers: {missing_ciks[:10]}{'...' if len(missing_ciks) > 10 else ''}")
    
    # Step 2: Prepare price data for SEC function (needs 'ticker' and 'period_end' columns)
    price_df = df[['symbol', 'date', 'close']].rename(columns={
        'symbol': 'ticker',
        'date': 'period_end',
        'close': 'price_close'
    }).copy()
    
    # Ensure timezone-naive datetimes (SEC data is timezone-naive)
    if price_df['period_end'].dt.tz is not None:
        price_df['period_end'] = price_df['period_end'].dt.tz_localize(None)
    
    # Step 3: Fetch quarterly fundamentals from SEC for each ticker
    fundamental_data_list = []
    successful_count = 0
    failed_count = 0
    
    for i, symbol in enumerate(tqdm(symbols, desc="Fetching SEC fundamentals")):
        if symbol not in cik_map:
            failed_count += 1
            continue
            
        print(f"  📊 Fetching SEC data for {symbol} ({i+1}/{len(symbols)})...")
        
        try:
            # Fetch quarterly fundamentals using SEC API
            cik = cik_map[symbol]
            
            # Wrap the fetch in try-except to catch division by zero and continue with partial data
            try:
                quarterly_df = fetch_quarterly_fundamentals(symbol, cik, price_df)
            except ZeroDivisionError as e:
                print(f"    ⚠️  {symbol}: Division by zero during calculation, attempting to recover...")
                # Try to get raw data without computed ratios
                import traceback
                error_line = traceback.format_exc()
                
                # Try alternative: fetch raw facts without ratios
                try:
                    facts = get_company_facts(cik)
                    df_q = _pull_quarterly_facts(facts, cik)
                    if not df_q.empty:
                        quarterly_df = df_q[df_q["end"] >= START_DATE].copy()
                        quarterly_df['ticker'] = symbol
                        # Add basic columns without computed ratios
                        quarterly_df['market_cap'] = np.nan
                        quarterly_df['pe_q'] = np.nan
                        quarterly_df['pb'] = np.nan
                        quarterly_df['book_to_market'] = np.nan
                        quarterly_df['pe_ttm'] = np.nan
                        quarterly_df['ps_ttm'] = np.nan
                        quarterly_df['pcf_ttm'] = np.nan
                        quarterly_df['dividend_yield_ttm'] = np.nan
                        print(f"    ⚠️  {symbol}: Recovered with raw data (no computed ratios)")
                    else:
                        print(f"    ❌ {symbol}: Could not recover - no data")
                        failed_count += 1
                        continue
                except Exception as e2:
                    print(f"    ❌ {symbol}: Recovery failed - {str(e2)}")
                    failed_count += 1
                    continue
            
            if quarterly_df.empty:
                print(f"    ⚠️  {symbol}: No SEC data available")
                failed_count += 1
                continue
            
            # Add symbol column (SEC function uses 'ticker')
            quarterly_df['symbol'] = symbol
            
            # Use 'filed' date as the availability date (when data became public)
            # This ensures point-in-time accuracy - no lookahead bias
            if 'filed' in quarterly_df.columns:
                quarterly_df['filed_date'] = pd.to_datetime(quarterly_df['filed'])
            else:
                # Fallback: use quarter end date + 60 days if filed date not available
                quarterly_df['filed_date'] = pd.to_datetime(quarterly_df['end']) + pd.Timedelta(days=60)
            
            # Ensure timezone-naive
            if quarterly_df['filed_date'].dt.tz is not None:
                quarterly_df['filed_date'] = quarterly_df['filed_date'].dt.tz_localize(None)
            
            fundamental_data_list.append(quarterly_df)
            successful_count += 1
            print(f"    ✅ {symbol}: {len(quarterly_df)} quarterly records")
            
        except KeyError as e:
            print(f"    ⚠️  {symbol}: Missing column {str(e)} - skipping")
            failed_count += 1
            continue
        except Exception as e:
            print(f"    ❌ {symbol}: {type(e).__name__}: {str(e)}")
            failed_count += 1
            continue
    
    if not fundamental_data_list:
        print("❌ No SEC fundamental data retrieved")
        return df
    
    # Step 4: Combine all fundamental data
    print(f"\n📊 Summary:")
    print(f"   ✅ Successful: {successful_count}")
    print(f"   ❌ Failed: {failed_count}")
    
    fundamentals_df = pd.concat(fundamental_data_list, ignore_index=True)
    print(f"\n   Total fundamental records: {len(fundamentals_df):,}")
    print(f"   Date range: {fundamentals_df['filed_date'].min()} to {fundamentals_df['filed_date'].max()}")
    
    # Step 5: Merge with price data using point-in-time logic
    # For each price date, use the most recent fundamental data that was filed BEFORE that date
    print("\n🔄 Merging with price data (point-in-time)...")
    
    def merge_fundamentals_asof(price_df, fund_df):
        """Merge fundamentals using asof merge for point-in-time accuracy"""
        result_list = []
        
        for symbol in tqdm(price_df['symbol'].unique(), desc="Merging by symbol"):
            # Get price and fundamental data for this symbol
            symbol_prices = price_df[price_df['symbol'] == symbol].copy().sort_values('date')
            symbol_funds = fund_df[fund_df['symbol'] == symbol].copy().sort_values('filed_date')
            
            if symbol_funds.empty:
                result_list.append(symbol_prices)
                continue
            
            # Use merge_asof to match each price date with the most recent filed fundamental
            merged = pd.merge_asof(
                symbol_prices,
                symbol_funds,
                left_on='date',
                right_on='filed_date',
                by='symbol',
                direction='backward',  # Use most recent filed data before price date
                tolerance=pd.Timedelta(days=365*2)  # Max 2 years old
            )
            
            result_list.append(merged)
        
        return pd.concat(result_list, ignore_index=True)
    
    df_with_fundamentals = merge_fundamentals_asof(df, fundamentals_df)
    
    # Step 6: Clean up columns
    # Drop helper columns
    cols_to_drop = ['ticker', 'filed_date', 'price_close', 'period_end', 'filed']
    cols_to_drop = [c for c in cols_to_drop if c in df_with_fundamentals.columns]
    df_with_fundamentals = df_with_fundamentals.drop(columns=cols_to_drop, errors='ignore')
    
    # Rename SEC columns to match expected format
    rename_map = {
        'end': 'quarter_end',
        'fy': 'fiscal_year',
        'fp': 'fiscal_period',
        'shares_out': 'shares_outstanding',
        'eps_diluted': 'eps',
        'operating_cash_flow': 'operating_cashflow',
    }
    df_with_fundamentals = df_with_fundamentals.rename(columns=rename_map)
    
    # Get list of fundamental columns
    fundamental_cols = [c for c in df_with_fundamentals.columns 
                       if c not in ['symbol', 'date', 'open', 'high', 'low', 'close', 'volume']]
    
    print(f"\n✅ SEC fundamental data merge complete!")
    print(f"   Fundamental fields added: {len(fundamental_cols)}")
    print(f"   Fields: {fundamental_cols}")
    
    return df_with_fundamentals

# Add SEC quarterly fundamental data
if stock_data is not None:
    print("🔄 Adding SEC quarterly fundamental data...")
    stock_data = add_sec_quarterly_fundamentals(stock_data)
    
    print(f"\n📊 Sample of data with SEC fundamentals:")
    print(stock_data.head(10))
    
    print(f"\n📋 SEC Fundamental columns added:")
    fundamental_columns = [col for col in stock_data.columns if col not in ['symbol', 'date', 'open', 'high', 'low', 'close', 'volume']]
    for i, col in enumerate(fundamental_columns):
        print(f"   {i+1:2d}. {col}")
else:
    print("❌ No stock data available. Please run the data collection cells first.")


🔄 Adding SEC quarterly fundamental data...
🚀 Adding SEC quarterly fundamental data...
📈 Adding fundamentals for 285 stocks...

📥 Fetching CIK mappings from SEC...
📥 Fetching SEC ticker-CIK mapping...
✅ Found 10142 ticker-CIK mappings
   ✅ Found CIKs for 284/285 tickers
   ⚠️  Missing CIK for 1 tickers: ['IMB']


Fetching SEC fundamentals:   0%|          | 0/285 [00:00<?, ?it/s]

  📊 Fetching SEC data for A (1/285)...


Fetching SEC fundamentals:   0%|          | 1/285 [01:14<5:53:48, 74.75s/it]

    ✅ A: 47 quarterly records
  📊 Fetching SEC data for AA (2/285)...
    ⚠️  AA: Division by zero during calculation, attempting to recover...


...

    ✅ ZS: 36 quarterly records
  📊 Fetching SEC data for ZTS (285/285)...


Fetching SEC fundamentals: 100%|██████████| 285/285 [9:01:13<00:00, 113.94s/it]


    ✅ ZTS: 43 quarterly records

📊 Summary:
   ✅ Successful: 275
   ❌ Failed: 10

   Total fundamental records: 11,588
   Date range: 2014-11-29 00:00:00 to 2025-11-29 00:00:00

🔄 Merging with price data (point-in-time)...


Merging by symbol: 100%|██████████| 285/285 [00:13<00:00, 20.71it/s]



✅ SEC fundamental data merge complete!
   Fundamental fields added: 26
   Fields: ['quarter_end', 'fiscal_year', 'fiscal_period', 'frame', 'year', 'quarter', 'revenue', 'net_income', 'eps', 'shares_outstanding', 'equity', 'operating_cashflow', 'dividends_per_share', 'market_cap', 'pe_q', 'pb', 'book_to_market', 'shares_final', 'revenue_ttm', 'operating_cash_flow_ttm', 'eps_diluted_ttm', 'dividends_per_share_ttm', 'pe_ttm', 'ps_ttm', 'pcf_ttm', 'dividend_yield_ttm']

📊 Sample of data with SEC fundamentals:
        date       open       high        low      close   volume symbol  \
0 2015-01-02  37.688386  37.807365  36.947064  37.120956  1529200      A   
1 2015-01-05  36.901310  37.029439  36.333880  36.425400  2041800      A   
2 2015-01-06  36.434540  36.626733  35.711522  35.857956  2080600      A   
3 2015-01-07  36.169143  36.434555  35.958645  36.333881  3359700      A   
4 2015-01-08  36.828074  37.505327  36.773160  37.422958  2116300      A   
5 2015-01-09  37.523643  37.5236

## Step 5.5: Calculate Additional Fundamental Ratios

Create derived ratios from SEC fundamental data for enhanced ML features.


In [ ]:
def add_calculated_ratios(df):
    """
    Add calculated fundamental ratios from SEC data.
    
    Creates profitability, valuation, growth, and quality metrics
    derived from base SEC fundamental fields.
    """
    print("🔢 Calculating additional fundamental ratios...")
    
    if df is None or len(df) == 0:
        print("❌ No data available")
        return df
    
    initial_cols = len(df.columns)
    
    # ===== Profitability Ratios =====
    print("   📊 Profitability ratios...")
    
    # Return on Equity (ROE) - Key profitability metric
    df['roe'] = safe_divide(df['net_income'], df['equity'])
    
    # Profit Margins
    df['profit_margin'] = safe_divide(df['net_income'], df['revenue'])
    df['profit_margin_ttm'] = safe_divide(df['net_income'] * 4, df['revenue_ttm'])
    
    # Operating Cash Flow Margin
    df['ocf_margin'] = safe_divide(df['operating_cashflow'], df['revenue'])
    df['ocf_margin_ttm'] = safe_divide(df['operating_cash_flow_ttm'], df['revenue_ttm'])
    
    # ===== Per-Share Metrics =====
    print("   📊 Per-share metrics...")
    
    # Book Value Per Share
    df['book_value_per_share'] = safe_divide(df['equity'], df['shares_outstanding'])
    
    # Revenue Per Share
    df['revenue_per_share'] = safe_divide(df['revenue'], df['shares_outstanding'])
    df['revenue_per_share_ttm'] = safe_divide(df['revenue_ttm'], df['shares_outstanding'])
    
    # Cash Flow Per Share
    df['cashflow_per_share'] = safe_divide(df['operating_cashflow'], df['shares_outstanding'])
    df['cashflow_per_share_ttm'] = safe_divide(df['operating_cash_flow_ttm'], df['shares_outstanding'])
    
    # ===== Dividend Ratios =====
    print("   📊 Dividend ratios...")
    
    # Payout Ratio - % of earnings paid as dividends
    df['payout_ratio'] = safe_divide(df['dividends_per_share_ttm'], df['eps_diluted_ttm'])
    
    # Dividend Coverage - how many times dividends are covered by earnings
    df['dividend_coverage'] = safe_divide(df['eps_diluted_ttm'], df['dividends_per_share_ttm'])
    
    # Cash Flow Payout Ratio
    df['cf_payout_ratio'] = safe_divide(
        df['dividends_per_share_ttm'] * df['shares_outstanding'], 
        df['operating_cash_flow_ttm']
    )
    
    # ===== Growth Ratios =====
    print("   📊 Growth metrics (YoY)...")
    
    # Sort by symbol and date for proper time series operations
    df = df.sort_values(['symbol', 'date'])
    
    # Revenue Growth (Year-over-Year = 4 quarters ago)
    df['revenue_growth_yoy'] = df.groupby('symbol')['revenue'].pct_change(periods=4)
    df['revenue_ttm_growth_yoy'] = df.groupby('symbol')['revenue_ttm'].pct_change(periods=4)
    
    # Earnings Growth (YoY)
    df['earnings_growth_yoy'] = df.groupby('symbol')['net_income'].pct_change(periods=4)
    
    # EPS Growth (YoY)
    df['eps_growth_yoy'] = df.groupby('symbol')['eps'].pct_change(periods=4)
    df['eps_ttm_growth_yoy'] = df.groupby('symbol')['eps_diluted_ttm'].pct_change(periods=4)
    
    # Cash Flow Growth (YoY)
    df['cashflow_growth_yoy'] = df.groupby('symbol')['operating_cashflow'].pct_change(periods=4)
    df['cashflow_ttm_growth_yoy'] = df.groupby('symbol')['operating_cash_flow_ttm'].pct_change(periods=4)
    
    # Quarter-over-Quarter Growth (optional, more volatile)
    df['revenue_growth_qoq'] = df.groupby('symbol')['revenue'].pct_change()
    df['earnings_growth_qoq'] = df.groupby('symbol')['net_income'].pct_change()
    
    # ===== Quality & Efficiency Metrics =====
    print("   📊 Quality & efficiency metrics...")
    
    # Earnings Quality (Cash Flow / Net Income) - higher is better
    df['earnings_quality'] = safe_divide(df['operating_cashflow'], df['net_income'])
    df['earnings_quality_ttm'] = safe_divide(df['operating_cash_flow_ttm'], df['net_income'] * 4)
    
    # Cash Conversion Rate (Operating CF / Revenue)
    df['cash_conversion'] = safe_divide(df['operating_cashflow'], df['revenue'])
    df['cash_conversion_ttm'] = safe_divide(df['operating_cash_flow_ttm'], df['revenue_ttm'])
    
    # Accruals Ratio (Earnings - Cash Flow) / Equity - lower is better
    df['accruals'] = safe_divide(df['net_income'] - df['operating_cashflow'], df['equity'])
    
    # ===== Valuation Metrics =====
    print("   📊 Valuation metrics...")
    
    # PEG Ratio (PE / Growth Rate) - only valid when growth is positive
    with np.errstate(divide='ignore', invalid='ignore'):
        df['peg_ratio'] = safe_divide(df['pe_ttm'], df['eps_ttm_growth_yoy'] * 100)
        # Cap extreme values
        df['peg_ratio'] = df['peg_ratio'].clip(-10, 10)
    
    # Price to Cash Flow (already have pcf_ttm, this is non-TTM version)
    df['price_to_cashflow'] = safe_divide(df['market_cap'], df['operating_cashflow'] * 4)
    
    # ===== Summary =====
    added_cols = len(df.columns) - initial_cols
    print(f"\n✅ Added {added_cols} calculated ratio features!")
    
    # List of new features
    new_features = [
        'roe', 'profit_margin', 'profit_margin_ttm', 'ocf_margin', 'ocf_margin_ttm',
        'book_value_per_share', 'revenue_per_share', 'revenue_per_share_ttm',
        'cashflow_per_share', 'cashflow_per_share_ttm',
        'payout_ratio', 'dividend_coverage', 'cf_payout_ratio',
        'revenue_growth_yoy', 'revenue_ttm_growth_yoy', 'earnings_growth_yoy',
        'eps_growth_yoy', 'eps_ttm_growth_yoy', 'cashflow_growth_yoy', 'cashflow_ttm_growth_yoy',
        'revenue_growth_qoq', 'earnings_growth_qoq',
        'earnings_quality', 'earnings_quality_ttm', 'cash_conversion', 'cash_conversion_ttm',
        'accruals', 'peg_ratio', 'price_to_cashflow'
    ]
    
    print(f"\n📋 New Features:")
    for i, feat in enumerate(new_features, 1):
        if feat in df.columns:
            non_null = df[feat].notna().sum()
            pct = (non_null / len(df) * 100)
            print(f"   {i:2d}. {feat:30s} - {non_null:8,} values ({pct:5.1f}%)")
    
    return df

# Apply calculated ratios
if stock_data is not None:
    print("🔄 Adding calculated fundamental ratios...")
    stock_data = add_calculated_ratios(stock_data)
    
    print(f"\n📊 Updated dataset shape: {stock_data.shape}")
    print(f"   Total columns: {len(stock_data.columns)}")
    print(f"   Total records: {len(stock_data):,}")
else:
    print("❌ No stock data available. Please run the data collection cells first.")


## Step 6: Data Validation & Quality Check

Validate the collected data for completeness, quality, and consistency.


In [17]:
def validate_data_quality(df):
    """Validate data quality and completeness"""
    print("🔍 Validating data quality and completeness...")
    
    if df is None or len(df) == 0:
        print("❌ No data to validate")
        return False
    
    validation_results = {}
    
    # Basic data info
    validation_results['total_records'] = len(df)
    validation_results['total_columns'] = len(df.columns)
    validation_results['unique_symbols'] = df['symbol'].nunique()
    validation_results['date_range'] = {
        'start': df['date'].min(),
        'end': df['date'].max()
    }
    
    # Check for missing values
    missing_values = df.isnull().sum()
    validation_results['missing_values'] = missing_values[missing_values > 0].to_dict()
    
    # Check for duplicate records
    duplicates = df.duplicated(subset=['symbol', 'date']).sum()
    validation_results['duplicate_records'] = duplicates
    
    # Check for negative prices/volumes
    negative_prices = (df[['open', 'high', 'low', 'close']] < 0).any().any()
    negative_volumes = (df['volume'] < 0).any()
    validation_results['negative_values'] = {
        'prices': negative_prices,
        'volumes': negative_volumes
    }
    
    # Check for zero volumes
    zero_volumes = (df['volume'] == 0).sum()
    validation_results['zero_volumes'] = zero_volumes
    
    # Check for extreme price movements (daily returns > 50%)
    df_sorted = df.sort_values(['symbol', 'date'])
    df_sorted['daily_return'] = df_sorted.groupby('symbol')['close'].pct_change()
    extreme_movements = (abs(df_sorted['daily_return']) > 0.5).sum()
    validation_results['extreme_movements'] = extreme_movements
    
    # Check data consistency (high >= low, etc.)
    inconsistent_high_low = (df['high'] < df['low']).sum()
    inconsistent_open_close = (df['open'] < 0).sum() + (df['close'] < 0).sum()
    validation_results['inconsistent_data'] = {
        'high_low': inconsistent_high_low,
        'negative_prices': inconsistent_open_close
    }
    
    # Print validation results
    print(f"\n📊 Data Quality Summary:")
    print(f"   📈 Total records: {validation_results['total_records']:,}")
    print(f"   📋 Total columns: {validation_results['total_columns']}")
    print(f"   🏢 Unique symbols: {validation_results['unique_symbols']}")
    print(f"   📅 Date range: {validation_results['date_range']['start']} to {validation_results['date_range']['end']}")
    print(f"   🔄 Duplicate records: {validation_results['duplicate_records']}")
    print(f"   📉 Zero volumes: {validation_results['zero_volumes']}")
    print(f"   ⚡ Extreme movements (>50%): {validation_results['extreme_movements']}")
    print(f"   ❌ Inconsistent high/low: {validation_results['inconsistent_data']['high_low']}")
    print(f"   ❌ Negative prices: {validation_results['inconsistent_data']['negative_prices']}")
    
    # Missing values summary
    if validation_results['missing_values']:
        print(f"\n⚠️  Missing values by column:")
        for col, missing_count in validation_results['missing_values'].items():
            missing_pct = (missing_count / len(df)) * 100
            print(f"   {col}: {missing_count:,} ({missing_pct:.1f}%)")
    else:
        print(f"\n✅ No missing values found!")
    
    # Overall quality assessment
    quality_score = 100
    if validation_results['duplicate_records'] > 0:
        quality_score -= 10
    if validation_results['zero_volumes'] > len(df) * 0.01:  # More than 1% zero volumes
        quality_score -= 5
    if validation_results['extreme_movements'] > len(df) * 0.001:  # More than 0.1% extreme movements
        quality_score -= 5
    if validation_results['inconsistent_data']['high_low'] > 0:
        quality_score -= 20
    if validation_results['inconsistent_data']['negative_prices'] > 0:
        quality_score -= 20
    
    validation_results['quality_score'] = quality_score
    
    print(f"\n🎯 Overall Quality Score: {quality_score}/100")
    
    if quality_score >= 90:
        print("✅ Excellent data quality!")
    elif quality_score >= 80:
        print("✅ Good data quality!")
    elif quality_score >= 70:
        print("⚠️  Acceptable data quality with some issues")
    else:
        print("❌ Poor data quality - needs attention")
    
    return validation_results

# Validate data quality
if stock_data is not None:
    validation_results = validate_data_quality(stock_data)
else:
    print("❌ No data available for validation")


🔍 Validating data quality and completeness...

📊 Data Quality Summary:
   📈 Total records: 669,825
   📋 Total columns: 33
   🏢 Unique symbols: 285
   📅 Date range: 2015-01-02 00:00:00 to 2024-12-30 00:00:00
   🔄 Duplicate records: 0
   📉 Zero volumes: 287
   ⚡ Extreme movements (>50%): 23
   ❌ Inconsistent high/low: 0
   ❌ Negative prices: 0

⚠️  Missing values by column:
   quarter_end: 26,208 (3.9%)
   fiscal_year: 318,204 (47.5%)
   fiscal_period: 318,204 (47.5%)
   frame: 315,423 (47.1%)
   year: 26,208 (3.9%)
   quarter: 26,208 (3.9%)
   revenue: 328,910 (49.1%)
   net_income: 207,450 (31.0%)
   eps: 168,658 (25.2%)
   shares_outstanding: 190,100 (28.4%)
   equity: 79,649 (11.9%)
   operating_cashflow: 245,593 (36.7%)
   dividends_per_share: 351,124 (52.4%)
   market_cap: 386,330 (57.7%)
   pe_q: 417,719 (62.4%)
   pb: 405,410 (60.5%)
   book_to_market: 405,410 (60.5%)
   shares_final: 147,607 (22.0%)
   revenue_ttm: 550,579 (82.2%)
   operating_cash_flow_ttm: 663,432 (99.0%)
   e

## Step 7: Data Persistence

Save all collected data to files for future use in Phase 2.


In [18]:
def save_research_data(df, successful_tickers, failed_tickers, validation_results):
    """Save all research data to files"""
    print("💾 Saving research data to files...")
    
    if df is None or len(df) == 0:
        print("❌ No data to save")
        return False
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    try:
        # Save main data as CSV
        csv_file = f'{data_dir}/sp500_stock_data_{timestamp}.csv'
        df.to_csv(csv_file, index=False)
        print(f"✅ Saved stock data to: {csv_file}")
        
        # Save as pickle for faster loading
        pkl_file = f'{data_dir}/sp500_stock_data_{timestamp}.pkl'
        df.to_pickle(pkl_file)
        print(f"✅ Saved stock data to: {pkl_file}")
        
        # Save metadata
        metadata = {
            'timestamp': timestamp,
            'data_info': {
                'total_records': len(df),
                'total_columns': len(df.columns),
                'unique_symbols': df['symbol'].nunique(),
                'date_range': {
                    'start': df['date'].min().strftime('%Y-%m-%d'),
                    'end': df['date'].max().strftime('%Y-%m-%d')
                },
                'columns': list(df.columns)
            },
            'tickers': {
                'successful': successful_tickers,
                'failed': failed_tickers,
                'total_requested': len(successful_tickers) + len(failed_tickers)
            },
            'validation': validation_results,
            'files': {
                'csv': csv_file,
                'pkl': pkl_file
            }
        }
        
        metadata_file = f'{data_dir}/sp500_metadata_{timestamp}.json'
        with open(metadata_file, 'w') as f:
            json.dump(metadata, f, indent=2, default=str)
        print(f"✅ Saved metadata to: {metadata_file}")
        
        # Save latest files (without timestamp for easy access)
        latest_csv = f'{data_dir}/sp500_stock_data_latest.csv'
        latest_pkl = f'{data_dir}/sp500_stock_data_latest.pkl'
        latest_metadata = f'{data_dir}/sp500_metadata_latest.json'
        
        df.to_csv(latest_csv, index=False)
        df.to_pickle(latest_pkl)
        
        with open(latest_metadata, 'w') as f:
            json.dump(metadata, f, indent=2, default=str)
        
        print(f"✅ Saved latest files:")
        print(f"   📄 CSV: {latest_csv}")
        print(f"   🗃️  PKL: {latest_pkl}")
        print(f"   📋 Metadata: {latest_metadata}")
        
        return True
        
    except Exception as e:
        print(f"❌ Error saving data: {str(e)}")
        return False

# Save all data
if stock_data is not None:
    save_success = save_research_data(stock_data, successful_tickers, failed_tickers, validation_results)
    
    if save_success:
        print(f"\n🎉 Phase 1 Complete!")
        print(f"   📊 Data collected and saved successfully")
        print(f"   💾 Ready for Phase 2: Feature Engineering & ML")
        print(f"   📁 Data directory: {data_dir}")
    else:
        print(f"\n❌ Phase 1 failed - data not saved properly")
else:
    print("❌ No data available to save")


💾 Saving research data to files...
✅ Saved stock data to: data/research/sp500_stock_data_20251023_055803.csv
✅ Saved stock data to: data/research/sp500_stock_data_20251023_055803.pkl
✅ Saved metadata to: data/research/sp500_metadata_20251023_055803.json
✅ Saved latest files:
   📄 CSV: data/research/sp500_stock_data_latest.csv
   🗃️  PKL: data/research/sp500_stock_data_latest.pkl
   📋 Metadata: data/research/sp500_metadata_latest.json

🎉 Phase 1 Complete!
   📊 Data collected and saved successfully
   💾 Ready for Phase 2: Feature Engineering & ML
   📁 Data directory: data/research


## Step 8: Data Loading Test

Test loading the saved data to ensure it can be properly retrieved for Phase 2.


In [19]:
def load_research_data(data_dir='data/research'):
    """Load previously saved research data"""
    print("📂 Loading previously saved research data...")
    
    try:
        # Try to load latest files
        latest_pkl = f'{data_dir}/sp500_stock_data_latest.pkl'
        latest_metadata = f'{data_dir}/sp500_metadata_latest.json'
        
        if os.path.exists(latest_pkl) and os.path.exists(latest_metadata):
            # Load data
            df = pd.read_pickle(latest_pkl)
            
            # Load metadata
            with open(latest_metadata, 'r') as f:
                metadata = json.load(f)
            
            print(f"✅ Successfully loaded data!")
            print(f"   📊 Records: {len(df):,}")
            print(f"   📋 Columns: {len(df.columns)}")
            print(f"   🏢 Symbols: {df['symbol'].nunique()}")
            print(f"   📅 Date range: {df['date'].min()} to {df['date'].max()}")
            print(f"   💾 Saved: {metadata['timestamp']}")
            
            return df, metadata
        else:
            print(f"❌ Latest data files not found in {data_dir}")
            return None, None
            
    except Exception as e:
        print(f"❌ Error loading data: {str(e)}")
        return None, None

# Test loading data
print("🧪 Testing data loading...")
loaded_data, loaded_metadata = load_research_data()

if loaded_data is not None:
    print(f"\n📊 Sample of loaded data:")
    print(loaded_data.head())
    
    print(f"\n📋 Data summary:")
    print(f"   Shape: {loaded_data.shape}")
    print(f"   Columns: {list(loaded_data.columns)}")
    
    print(f"\n✅ Data loading test successful!")
    print(f"   🚀 Ready to proceed to Phase 2")
else:
    print(f"\n❌ Data loading test failed")
    print(f"   🔄 Please run Phase 1 data collection first")


🧪 Testing data loading...
📂 Loading previously saved research data...
✅ Successfully loaded data!
   📊 Records: 669,825
   📋 Columns: 33
   🏢 Symbols: 285
   📅 Date range: 2015-01-02 00:00:00 to 2024-12-30 00:00:00
   💾 Saved: 20251023_055803

📊 Sample of loaded data:
        date       open       high        low      close   volume symbol  \
0 2015-01-02  37.688386  37.807365  36.947064  37.120956  1529200      A   
1 2015-01-05  36.901310  37.029439  36.333880  36.425400  2041800      A   
2 2015-01-06  36.434540  36.626733  35.711522  35.857956  2080600      A   
3 2015-01-07  36.169143  36.434555  35.958645  36.333881  3359700      A   
4 2015-01-08  36.828074  37.505327  36.773160  37.422958  2116300      A   

  quarter_end fiscal_year fiscal_period frame    year  quarter revenue  \
0  2014-10-31         NaN           NaN   NaN  2014.0      4.0     NaN   
1  2014-10-31         NaN           NaN   NaN  2014.0      4.0     NaN   
2  2014-10-31         NaN           NaN   NaN  2014.

---

# 🎉 Phase 1 Complete!

## ✅ What We've Accomplished:

1. **📊 Data Collection**: Fetched real S&P 500 stock data (OHLCV) for 250+ tickers
2. **💰 Fundamental Data**: Added time-varying fundamental data every 5 months
3. **🔍 Quality Validation**: Validated data completeness and quality
4. **💾 Data Persistence**: Saved all data to CSV/PKL files for future use
5. **🧪 Loading Test**: Verified data can be properly loaded

## 🚀 Next Steps:

**Phase 2: Feature Engineering & ML** will include:
- Load saved data (skip re-fetching)
- Implement 80+ features from the paper
- Create momentum/reversal targets
- Train ML models (RF, GB, AdaBoost, etc.)
- Backtesting and performance evaluation
- Results analysis and visualization
